# Analytical and symbolic reconstructors at one wavelength

**Goal.** A reconstructor that is an analytical expression of moderate
complexity: a function of the field sampled at the lantern's 19 output
positions that returns each petal piston, without the hundreds to thousands
of degrees of freedom of the polynomial models, or the tens of thousands of
the MLPs. This is the 800 nm, +-400 nm (+-lambda/2) case of `workbench.ipynb` Part A, on
the 50000-sample dataset (`data/ml/single_800nm_50k.npz`): fit on rows
0-36000, model selection on 36000-40000, validation on 40000-50000.

The answer, in short:

- **The forward map is exactly analytical.** With piston-only aberrations
  and monochromatic light, every output power is a Hermitian quadratic form
  in the six petal phasors. A closed form with 619 physical constants
  reproduces the camera signals to numerical precision (section 1).
- **Its inversion is an excellent reconstructor** (section 7). Gauss-Newton
  converges within 5 iterations for 99.95% of cases from a good first
  estimate, and a staged procedure solves every case. The result is exact
  without noise, 0.9 nm RMS with 1% noise, and has no gross errors up to
  3% noise. The 619 constants can be calibrated from about 100 known
  configurations.
- **Explicit expressions over the full +-400 nm range are much harder**
  (sections 3-4). The inverse of the trigonometric map is complicated. Even
  with symmetry-adapted features, polynomials need about 200 terms per
  phasor for 10-24 nm, and symbolic regression (PySR, Operon) stays near
  80 nm at moderate complexity.
- **Over smaller ranges explicit expressions work** (sections 5-6). At
  +-100 nm PySR finds expressions with 4-8 constants and about 40 nodes
  that reach 2.6-4.2 nm, against 6-8 nm for a linear map. Below +-50 nm
  even the linear reconstructor is accurate to 1-2%.
- **The limit is the aberrations the model leaves out** (section 7e).
  Low-order residuals (tip/tilt, focus, astigmatism) leak into the piston
  estimates at about 0.6 nm per nm RMS, for the analytical reconstructor and
  the MLP alike.

In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import pandas as pd
import ml_reconstruct as ml
from ml_reconstruct import forward as fw, symbolic as sy, petals
from ml_reconstruct.simulate import bp
from sklearn.linear_model import RidgeCV

LAM, FIT, ES, VA = 800.0, slice(0, 36000), slice(36000, 40000), slice(40000, 50000)
cache = ml.ModelCache("data/ml/cache_symbolic")

In [ ]:
ds = ml.LanternDataset.load("data/ml/single_800nm_50k.npz")
S, P, xy = ds.signals, ds.coeff, ds.core_xy
p800 = bp.get_simulation_parameters(3, 0.8)
ifunc = bp.load_ifunc(p800["ifunc_file"]); bp.center_ifunc_mask(ifunc)
geom = petals.petal_geometry(ifunc.mask_inf_func, np.asarray(ifunc.influence_function))
linear = cache("linear-pistons", lambda: RidgeCV(alphas=np.logspace(-6, 2, 9)).fit(S[FIT], P[FIT]))
print(f"{len(S)} samples, {S.shape[1]} cores; linear reconstructor: RMSE "
      f"{ml.rmse_nm(linear.predict(S[VA]), P[VA]).round(1)} nm")

## 1. The forward map is exactly analytical

The field at an output core is a linear function of the input field, and
with piston-only aberrations the input field is a sum over the six petals of
exp(i phi_j) times a fixed petal pattern. So the core power is

N_c = a_c + sum_{j<k} [A_cjk cos D_jk + B_cjk sin D_jk],  with  D_jk = 2 pi (p_j - p_k) / lambda

(the reference petal has p = 0). That is linear in 30 trigonometric features
z, with 31 constants per core. The camera signals are normalised by the
coupled power, which is a form of the same kind, so

s_c = N_c / (1 + q . z),

with 19 x 31 + 30 = 619 constants. Given the pistons it is linear in the
constants, and one least-squares solve fits it.

This is not a shortcut that applies the piston phases at the lantern input.
Every signal here comes from the full simulation: pupil phase -> zero-padded
FFT -> PSF field at the lantern input -> projection on the lantern modes ->
cbeam propagation -> camera windows. The closed form follows from linearity:

- **PSF field:** it is the sum over the petals of exp(i phi_j) times the
  fixed focal-plane field E_j(u) of petal j alone.
- **Lantern output:** the lantern is linear in its input field, so its
  output is the sum of exp(i phi_j) times petal j's response O_j(x).
- **Signals:** a camera window integrates |output|^2, which gives the sum
  over j, k of exp(i (phi_j - phi_k)) times the integral of O_j O_k*.

The constants therefore absorb the petal PSFs, the lantern's mode projection
and propagation, and the camera windows. Here they are fitted to simulated
data; they could equally be computed directly, by propagating the six
single-petal PSF fields. The form is exact because the aberrations are
pistons. Other aberrations change the petal fields themselves, which section
7e tests.

In [ ]:
chan = ml.LanternDataset.load("data/ml/spectral_41wl.npz"); w800 = int(np.argmin(np.abs(chan.wavelengths_nm - LAM)))
models = {"camera signals, ratio form (619)": (fw.fit_forward(P[:10000], S[:10000], LAM), S, P),
          "camera signals, numerator only (589)": (fw.fit_forward(P[:10000], S[:10000], LAM, normalised=False), S, P),
          "channel powers, +-5 um (589)": (fw.fit_forward(chan.coeff[:10000], chan.signals[:10000, w800], LAM,
                                                           normalised=False), chan.signals[:, w800], chan.coeff)}
for name, (m, Sx, Px) in models.items():
    r = Sx[VA] - m(Px[VA])
    print(f"{name:40s} relative residual RMS on the validation rows: {np.sqrt(np.mean(r ** 2)) / Sx[VA].std():.1e}")
model = models["camera signals, ratio form (619)"][0]

The ratio form fits to numerical precision. The channel powers (fraction of
the incident light, with no normalisation) are exactly the numerator form.
So the lantern at one wavelength is completely described, for pistons, by
how strongly each core interferes each pair of petals:

In [ ]:
fw.plot_coefficients(model)

## 2. Symmetry-adapted features

For explicit expressions, the representation of the input matters more than
the engine. Two choices:

- **Inputs.** The 19 cores lie on rings of a hexagonal grid: the centre, and
  three rings of 6 (at a, sqrt(3) a and 2 a). The angular harmonics of each
  ring, sum_c s_c exp(-i m theta_c) for m = 0..3, re-express the 19 signals
  as 19 numbers ordered by symmetry: moments of the output field. For piston
  k they are taken in the frame of the bisector of petal k and the
  reference, oriented so that petal k is on the positive side. Their real
  parts are even, and their imaginary parts odd, under the mirror that swaps
  petal k and the reference. Pistons related by the mirror (0 and 4, 1 and
  3) then see the same problem, and can share one expression.
- **Targets.** Each piston's phasor (sin, cos)(2 pi p_k / lambda), decoded
  with atan2. This is unique within +-lambda/2, and it saves the expression
  from having to contain an arctangent.

In [ ]:
ring, theta = sy.rings(xy)
print("cores per ring:", np.bincount(ring))
for k in range(5):
    a, o = sy.pair_frame(geom, k)
    print(f"piston {k}: petal at {np.degrees(sy.petal_angles_mesh(geom)[k]):7.1f} deg, frame {np.degrees(a):7.1f} deg, "
          f"orientation {o:+.0f}")
F_k = {k: sy.harmonic_features(S, xy, *sy.pair_frame(geom, k)) for k in range(5)}
print("features:", F_k[0][1])

## 3. Explicit polynomial models of the phasors

How complex does an explicit map have to be? Polynomials in the harmonic
features, fitted to the phasors of each piston, with all degree-1 terms
(ridge), or a budget of K terms per phasor chosen by OMP. "dof" counts the
fitted constants of the two phasors of one piston.

In [ ]:
configs = [("linear", 1, None), ("quadratic, K = 20", 2, 20), ("quadratic, K = 60", 2, 60),
           ("cubic, K = 200", 3, 200)]
rows = []
for name, deg, K in configs:
    for k in range(5):
        F, T = F_k[k][0], sy.phasor_targets_k(P, k, LAM)
        m = cache(f"phasor-k{k}-d{deg}-K{K}", lambda: sy.fit_phasor_poly(F[FIT], T[FIT], deg, K))
        pr = sy.predict_phasor_poly(m, F[VA])
        e = sy.decode(pr[:, 0], pr[:, 1], LAM) - P[VA, k]
        rows.append(dict(model=name, piston=k, dof=sy.phasor_poly_dof(m), rmse=np.sqrt(np.mean(e ** 2))))
poly = pd.DataFrame(rows).pivot(index="model", columns="piston", values=["rmse", "dof"]).round(1)
display(poly.loc[[c[0] for c in configs]])

For reference, the best cubic model of `workbench.ipynb` (Part A, A4): OMP
with 1282 terms per piston on the degree-3 monomials of the standardised
Zernike projection. It regresses the pistons directly, was trained on the
8000 training rows of the 10k dataset (the first 8000 rows here), and is
scored on the same 10000 validation rows as everything else in this
notebook:

**The K = 1282 of the reference model** is the saturated OMP size on the core-signal
cubic basis (1281 non-zero monomials + the intercept, i.e. 1282 parameters per piston);
`workbench.ipynb` Part A contains the sweep that locates it.

In [ ]:
p_wb, dof_wb = sy.workbench_fiber_omp(S[:8000], P[:8000], S[VA])
e_wb = np.abs(p_wb - P[VA])
print(f"workbench cubic OMP (core signals), K = 1282: RMSE {ml.rmse_nm(p_wb, P[VA]).round(2)} nm, "
      f"median {np.median(e_wb):.2f} nm, 99th percentile {np.percentile(e_wb, 99):.1f} nm; "
      f"{dof_wb} constants per piston")


A linear map of the moments is far off (80-130 nm), and the error falls only
slowly with the number of terms: about 200 terms per phasor give 10-24 nm,
and the workbench's 1282 terms per piston 8.4 nm (6406 constants).
The inverse of the trigonometric forward map is genuinely complicated. Each
phasor depends on the others through the interference terms of all the
petal pairs.

## 4. Symbolic regression

`scripts/symbolic_sr.py` (run by `scripts/run_symbolic_sr.sh`) searches for
explicit expressions of the 19 harmonic features for the sine and cosine of
each class of piston:

- **classes:** opposite (piston 2), adjacent (0 and 4 pooled) and
  next-but-one (1 and 3 pooled);
- **operators:** +, -, x, / and the square;
- **data:** 10000 fit rows per piston;
- **engines:** **Operon** (pyoperon: multi-objective, the whole
  accuracy-vs-length front) and **PySR** (its hall of fame over
  complexities, 10 minutes per target).

For each class the sine and cosine expressions are paired at every total
complexity, and the pair with the best held-out piston RMSE is kept. The
front is scored on the validation rows. "constants" counts the fitted
numbers in both expressions.

In [ ]:
SR_DIR = "data/ml/symbolic"
fronts = {}
for engine in ("operon", "pysr"):
    res = sy.load_sr(SR_DIR, engine)
    for cls in sy.CLASSES:
        if (cls, "sin") in res and (cls, "cos") in res:
            fronts[f"{engine}, {cls}"] = sy.sr_front(res, cls, LAM)
for label, f in fronts.items():
    print(label); display(f[["complexity", "constants", "es_rmse", "val_rmse", "val_median"]].round(2).tail(8))

In [ ]:
refs = {f"cubic K=200, piston {k}": poly.loc["cubic, K = 200", ("rmse", k)] for k in (0, 2)}
refs.update({f"quadratic K=60, piston 2": poly.loc["quadratic, K = 60", ("rmse", 2)],
             "linear, piston 2": poly.loc["linear", ("rmse", 2)],
             "workbench cubic K=1282 (1282 constants/piston)": ml.rmse_nm(p_wb, P[VA]).mean()})
sy.plot_sr_fronts(fronts, refs)

In [ ]:
for label, f in fronts.items():
    if len(f):
        r = f[f.complexity <= 60].iloc[-1] if (f.complexity <= 60).any() else f.iloc[0]
        print(f"{label}: complexity {r.complexity}, {r.constants} constants, validation RMSE {r.val_rmse:.1f} nm")
        print(f"   sin = {r.sin_expr}\n   cos = {r.cos_expr}\n")

Over the full +-400 nm range neither engine finds a compact inverse. At
complexity 60-100 both stay near 80 nm (R^2 0.5-0.9 on the phasors), barely
better than the linear map of section 3 and worse than its 60-term
quadratic polynomials. With 19 measurements and an inverse that undoes a
trigonometric map folding over +-0.75 pi of phase, there is no short closed
form to find.

## 5. Where explicit expressions work: the capture range

The difficulty is the range. Over +-400 nm (+-lambda/2) the phases reach +-pi and the
forward map folds strongly. Near the flat wavefront it is close to linear.
The exact forward model of section 1 is a noise-free simulator for any
range, so it can show how compact explicit reconstructors fare as the range
shrinks. The models: linear in the 19 signals, and sparse quadratic (OMP,
K terms per piston), fitted to the pistons directly. A closed-loop SCAO
petal sensor sees residuals of tens of nm, so the small ranges are the
relevant ones there.

In [ ]:
scan = sy.range_scan(model, [25, 50, 100, 200, 400])
display(scan.round(3))

Up to +-50 nm a linear reconstructor is already accurate to about 1-2% of
the range, and a quadratic correction with 30 terms per piston improves it
3-4 times. Beyond about +-100 nm explicit low-order expressions fail.
Acquisition over the full range needs the inversion of section 7.

## 6. Symbolic regression at +-100 nm

At +-100 nm the linear reconstructor is clearly imperfect (about 6 nm), but
a compact nonlinear correction should exist. Here symbolic regression
regresses the piston directly: same features and classes as in section 4,
noise-free data from the forward model (`scripts/symbolic_sr.py --range
100`).

In [ ]:
fronts100 = {}
for engine in ("operon", "pysr"):
    for cls, r in sy.load_sr_direct(SR_DIR, engine, 100).items():
        fronts100[f"{engine}, {cls}"] = sy.sr_front_direct(r)
for label, f in fronts100.items():
    print(label); display(f[["complexity", "constants", "es_rmse", "val_rmse"]].round(2).tail(8))

In [ ]:
row100 = scan[scan.range_nm == 100].iloc[0]
refs100 = {"linear (all pistons)": row100.filter(like="linear").iloc[0],
           "quadratic K=30": row100.filter(like="quadratic K=30").iloc[0]}
sy.plot_sr_fronts(fronts100, refs100)
for label, f in fronts100.items():
    if len(f):
        r = f[f.complexity <= 40].iloc[-1] if (f.complexity <= 40).any() else f.iloc[0]
        print(f"{label}: complexity {r.complexity}, {r.constants} constants, RMSE {r.val_rmse:.2f} nm\n   p = {r.expression}\n")

At +-100 nm both engines find compact expressions:

| class | PySR | linear map of the features (20 constants) |
|---|---|---|
| opposite (piston 2) | 2.6 nm (8 constants, complexity 36) | 5.8 nm |
| next-but-one (1, 3) | 2.7 nm (6 constants, complexity 39) | 5.8 nm |
| adjacent (piston 0 alone) | 4.2 nm (4 constants, complexity 40) | 8.4 nm |

That is as good as the sparse quadratic polynomials (3.1 nm on average, 155
constants for the 5 pistons). Operon reaches similar errors with more
constants. The expressions are polynomials of low degree in a few of the
harmonic moments: mostly the m = 1 and m = 2 moments of the rings, which
carry the odd (sign-sensitive) part of the response.

The pooled "adjacent" class (pistons 0 and 4 together) fails (about 25 nm),
and that is a flaw of these features, not of the physics. The lantern is
mirror-symmetric to about 1%. But the m = 3 harmonic of the 6-core rings
keeps only its real part, which is close to zero and flips sign between the
two adjacent pistons' frames, while the information sits in the imaginary
part that was dropped. Fitted alone, piston 0 behaves like the other
classes. The next-but-one class pools correctly.

## 7. The analytical reconstructor: invert the forward model

The forward map s(p) is an exact closed form, so the reconstructor can be its
inversion: minimise |s - s(p)|^2 over the 5 pistons by Gauss-Newton,

p <- p + (J^T J)^-1 J^T (s - s(p)),     J = ds/dp (analytic, 19 x 5),

from a first estimate. Each iteration costs one evaluation of the closed
form and of its Jacobian, and a 5 x 5 solve: a few thousand floating-point
operations. The model has 619 physical constants.

### 7a. The closed form

For every output core c, with D_jk = 2 pi (p_j - p_k) / lambda and petal 5
(the reference) at p_5 = 0:

In [ ]:
import sympy as sp
eq, p_sym, lam_sym = fw.sympy_generic()
display(eq)

The fitted constants for the central core and for one core of the first
ring, keeping the six largest terms of the numerator and of the denominator
(the numerator relative to the core's constant term; pistons in nm). With
all the terms, the sympy expression reproduces the model to machine
precision.

In [ ]:
for core in (0, 3):
    display(fw.sympy_forward(model, core, n_terms=6))
full = sp.lambdify(p_sym, fw.sympy_forward(model, 3, digits=15).rhs, "numpy")
print("full sympy expression of core 3 vs. the model:",
      f"{np.abs(full(*P[VA][:100].T) - model(P[VA][:100])[:, 3]).max() / S[:, 3].max():.1e} (relative)")

The central core interferes the petal pairs only through cosines. It is
blind to the sign of the pistons, as its symmetry requires. The off-axis
cores carry the sine terms, and with them the sign information.

### 7b. Convergence: how many iterations?

From three first estimates: the linear reconstructor (100 constants), the
cubic phasor polynomial of section 3, and the 36000-sample MLP ensemble of
`workbench.ipynb` Part C. The first estimate brings the case into the
right basin; Gauss-Newton then converges quadratically.

In [ ]:
from ml_reconstruct.notebook import rescore_mlp
p_cubic = np.stack([sy.decode(*sy.predict_phasor_poly(cache(f"phasor-k{k}-d3-K200", None), F_k[k][0][VA]).T, LAM)
                    for k in range(5)], 1)
p_mlp = rescore_mlp([f"800nm-50k-d4w256-cos3e-4-e2000-n36000-s{s}" for s in range(5)], "data/ml/mlp_hpo", ds, VA, 36000)
starts = {"linear": linear.predict(S[VA]), "cubic phasor poly": p_cubic, "MLP ensemble": p_mlp}
traces = {name: fw.gn_trace(model, S[VA], p0, P[VA]) for name, p0 in starts.items()}
display(fw.convergence_table(traces).round(2))
fw.plot_convergence(traces)

### 7c. The reconstructor as a procedure, and its cost

A single start leaves a few cases in the wrong basin, but the residual
|s - s(p)| flags them: it is about 1e-11 when converged and about 1e-3
otherwise. So the reconstructor is a procedure (`forward.solve_staged`):

1. 5 iterations from the first estimate;
2. for the flagged cases, 8 iterations each from the estimate's twin T p,
   its sign flip -p, -T p and the flat wavefront (the usual wrong basins,
   `piston2.ipynb`);
3. random restarts, 8 iterations each, for whatever is left.

The table gives the share of cases each stage solved, and the worst-case
number of iterations.

In [ ]:
T = petals.twin_matrix(geom["opposite"], geom["ref"])
rows = []
for name, p0 in starts.items():
    Ph, res, it, stage = fw.solve_staged(model, S[VA], p0, T, tol=1e-8)
    e = np.abs(Ph - P[VA]).max(1)
    rows.append(dict(start=name, **{f"solved in stage {k} %": 100 * np.mean(stage == k) for k in range(4)},
                     iterations_p99=np.percentile(it, 99), iterations_max=it.max(), max_error_nm=e.max()))
display(pd.DataFrame(rows).round(3))
print(f"one case, 5 iterations, numpy: {fw.time_single_case(model, S[40000], starts['linear'][0]):.0f} us")

**Convergence.**

- **From the MLP ensemble:** 94% of cases are below 0.01 nm after 1
  iteration, 99.8% after 2 and 99.95% after 5.
- **From the cubic phasor polynomial:** 99.4% after 3 iterations.
- **From the linear reconstructor:** only 80% after 5, because a tenth of its
  estimates start in the wrong basin.

**The staged procedure solves every case** (no case is left in stage 3):

- **From the MLP ensemble:** 99.94% of cases need only the first 5
  iterations, and the worst case 69.
- **From the linear reconstructor:** 13.7% need random restarts, and the
  worst case about 1400 iterations.

**Cost.** One iteration costs about 75 us for a single case in numpy (Python
overhead), and about 4 us per case when 10000 cases are batched. In compiled
code it is a few thousand floating-point operations.

### 7d. Noise

Gaussian noise of standard deviation (level x mean signal) on every core,
inverted with `forward.invert` (the first estimate plus 32 random starts,
keeping the lowest residual).

In [ ]:
levels = [0.0, 1e-4, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1]
noise = {"linear start": fw.noise_scan(model, S[VA], P[VA], lambda Sx: linear.predict(Sx), levels)}
display(noise["linear start"])
fw.plot_noise_scan(noise)

The error grows linearly with the noise, about 0.9 nm RMS per 1% of noise,
up to 10% noise (9.3 nm, 0.08% of cases off by more than 50 nm). At 30% the
inversion breaks down: 27% gross errors, as the noise starts to move the
lowest residual into other basins.

### 7e. Aberrations the model does not describe

Pistons only enter the forward model. `scripts/symbolic_residuals.py`
simulates 2000 configurations with random pistons in +-400 nm plus random
low-order residuals (tip, tilt, focus, astigmatism) of 0-100 nm RMS in total,
measured exactly like the dataset (the same core centres). Each is
reconstructed by inverting the forward model, starting from the MLP
ensemble plus 32 random starts, and, for comparison, by the MLP ensemble
itself, also trained on pistons only. "model misfit" is the RMS difference
between the signals and the forward model at the true pistons, relative to
the signals' spread.

In [ ]:
from ml_reconstruct.notebook import rescore_mlp_on
mlp_ids = [f"800nm-50k-d4w256-cos3e-4-e2000-n36000-s{s}" for s in range(5)]
rows = []
for level in (0, 5, 10, 25, 50, 100):
    with np.load(f"data/ml/symbolic/residuals_lo{level:g}.npz") as f:
        Sr, Cr = f["signals"], f["coeff"][:, :5]
    Pm = rescore_mlp_on(mlp_ids, "data/ml/mlp_hpo", ds, 36000, Sr)
    Ph, _, _ = fw.invert(model, Sr, Pm, n_starts=32)
    rows.append(dict(low_order_rms_nm=level, model_misfit=np.sqrt(np.mean((Sr - model(Cr)) ** 2)) / S.std(),
                     **{f"analytic {k}": v for k, v in fw.error_summary(Ph, Cr).items()},
                     **{f"MLP {k}": v for k, v in fw.error_summary(Pm, Cr).items()}))
display(pd.DataFrame(rows).round(3))

Aberrations outside the model are the real limit. Low-order residuals leak
into the piston estimates at about 0.6 nm of median piston error per nm RMS
of residual: 3 nm at 5 nm RMS, 15 nm at 25 nm RMS. Gross errors (more than
50 nm) affect 4% of cases at 25 nm RMS and most of them at 50 nm. The MLP, trained on pistons only, degrades in the same way. This is a
property of piston-only reconstructors at one wavelength, not of the
inversion.

Two ways forward:

- **Keep the low-order modes small.** The SCAO loop has to correct them well
  below about 5 nm RMS.
- **Model them.** The closed form extends to them: to first order in small
  low-order coefficients c_m, petal j's focal-plane field is
  E_j + i sum_m c_m E_jm. The input is then still a fixed linear
  combination of petal-phasor-weighted patterns, and the signals a quadratic
  form in a larger set of variables. This is a proposal, not tested here.

### 7f. Calibration: how many known configurations?

The 619 constants are fitted from known configurations. The table shows the
model fitted on the first n configurations, with 0.1% noise on both the
calibration and the test signals.

In [ ]:
calib = fw.calibration_scan(P[:20000], S[:20000], [40, 80, 160, 320, 1000, 5000], P[VA], S[VA],
                            lambda Sx: linear.predict(Sx), LAM, noise=1e-3)
display(calib)

With 0.1% noise, 40 known configurations already give 0.3 nm, and 160
reach the noise floor (0.1 nm). On a bench, the constants could be
calibrated from a short sequence of known piston patterns applied with M4,
or computed from the six single-petal PSF fields.

## 8. Can the symbolic search do better?

Two things were tried beyond sections 4 and 6:

- **An unrolled Newton.** An explicit first estimate followed by a fixed
  number of Gauss-Newton steps on the closed form is itself an explicit
  formula: no stopping rule, no restarts.
- **Template symbolic regression.** PySR searches theta = atan2(f, g) and
  scores the decoded phase with a periodic loss, instead of fitting sine
  and cosine separately.

### 8a. Explicit start + fixed Gauss-Newton steps

In [ ]:
starts9 = {"linear (100 constants)": linear.predict(S[VA]),
           "phasor poly K=60 (610)": np.stack([sy.decode(*sy.predict_phasor_poly(
               cache(f"phasor-k{k}-d2-K60", None), F_k[k][0][VA]).T, LAM) for k in range(5)], 1),
           "phasor poly K=200 (2010)": p_cubic, "workbench cubic K=1282 (6406)": p_wb}
display(fw.unrolled_newton_table(model, S[VA], P[VA], starts9).round(3))

**Two steps from the workbench's cubic model put 99.97% of the 10000
validation cases within 1 nm** (RMSE 0.02 nm), and three put every case
there, exactly.
Its estimates never start in the wrong basin, unlike the MLP's rare
outliers. More compact starts (the 60-term phasor polynomials) converge for
most cases (the median drops to about 0.01 nm after two steps), but
2-10% of their estimates start in a wrong basin, and those hold the RMSE
near 40 nm.

So there is a good explicit formula over the full range:
p = G(G(G(p_0(s)))), with p_0 the cubic polynomial and G one Gauss-Newton
step. It is exact
in practice, but has 6406 + 619 constants, so it is not compact. The
compactness limit is the first estimate: it only has to land in the right
basin, but every case must land there.

### 8b. Template symbolic regression: theta = atan2(f, g)

`scripts/symbolic_template.py` (run by `scripts/run_symbolic_template.sh`)
searches f and g jointly, 15 minutes per class, scoring the phase with the
periodic loss 1 - cos(theta_hat - theta). This is the quantity that matters,
whereas section 4 fitted sine and cosine separately. Classes: the opposite
piston, and piston 0 alone.

In [ ]:
fronts_t = {}
for cls in ("opposite", "adjacent-0"):
    path = f"{SR_DIR}/pysr-template_{cls}.npz"
    if os.path.exists(path):
        with np.load(path) as f:
            fronts_t[f"template, {cls}"] = sy.sr_front_template({k: f[k] for k in f.files}, LAM)
for label, f in fronts_t.items():
    print(label); display(f[["complexity", "constants", "es_rmse", "val_rmse"]].round(2).tail(8))
sy.plot_sr_fronts({**{k: v for k, v in fronts.items() if "opposite" in k or "next" in k}, **fronts_t}, refs)

Scoring the decoded phase directly helps, but only modestly. At complexity
up to about 60 (around 30 constants), the template reaches 72.5 nm for the
opposite piston and 73.1 nm for piston 0, against about 80 nm when sine and
cosine are fitted separately (section 4). That is still far from the
polynomials, let alone the inversion.

### 8c. Assessment

Over the full +-400 nm range, explicit expressions of moderate complexity
have stopped near 70-80 nm. The plain search, Operon, the atan2 template and
the symmetry-adapted features all end up there. Polynomials need thousands of
terms for below 10 nm. The limit looks structural, not a matter of search
effort or library:

- **There is no compact inverse.** The inverse undoes a trigonometric map
  that folds over +-0.75 pi of phase. Near the folds (`piston2.ipynb`) it
  changes quickly, and any smooth compact expression is wrong there.
  Section 5 shows the error falling quickly as the range shrinks, which is
  what a folding map predicts.
- **More search gives small gains.** Fitting the right quantity (the
  template) and running longer moved the front from about 80 to about 72 nm.
  Hours instead of minutes, or a larger maximum size, would improve it
  further, but not by the factor of 10 or more needed to compete.
- **Other libraries are unlikely to change this.**
  - *Genetic programming* (gplearn) is slower than PySR and Operon, which
    are among the strongest engines on the SRBench benchmarks.
  - *Deep symbolic optimisation* (DSO / uDSR) is competitive at a few
    variables, but not better in 19 dimensions.
  - *Pretrained transformer regressors* (E2E, NeSymReS) handle at most
    about 10 input variables, against 19 here.
  - *AI Feynman* looks for separability and symmetries; this inverse is not
    separable.
  - *SISSO and FFX* do sparse regression over generated features, which the
    OMP polynomials of section 3 already approximate.

  None of them can find a short closed form that does not exist.

What does work, and could be pursued:

1. **Smaller ranges.** In closed loop, compact expressions are good (section
   6: 4-8 constants, 2.6-4.2 nm at +-100 nm), and a linear map is enough
   below +-50 nm. A longer search there would likely shrink them further.
2. **An explicit start plus a few Newton steps** (section 8a). The start
   only has to land in the right basin. A compact symbolic start of about
   70 nm does not, for every case. But the staged procedure of section 7c
   detects and fixes its failures from the residual, so "8-constant start +
   Gauss-Newton with restarts" works, at the cost of restarts for a minority
   of cases.
3. **Several wavelengths.** Each wavelength has its own closed form, and
   wavelengths that unfold each other's folds may make the joint inverse
   simpler. That is untested here.

**Recommendation.** Use the closed form for the physics and the inversion,
and compact symbolic expressions for the small-residual, closed-loop
regime. Searching harder for a single compact explicit formula over the full
+-400 nm range is unlikely to pay off.

## 9. Conclusions

**The closed form.** At one wavelength and for petal pistons, the lantern's
response is an exact closed form. Each output signal is a ratio of two
linear combinations of cos and sin of the 15 pairwise petal phase
differences, with 619 constants in all (section 7a). It follows from the
linearity of the PSF and of the lantern in the petal phasors, and it
reproduces the full simulation to numerical precision.

**The reconstructors**, ranked:

| reconstructor | constants | +-400 nm | +-100 nm |
|---|---|---|---|
| inversion of the closed form (Gauss-Newton) | 619 (+ first estimate) | exact; 0.9 nm at 1% noise | exact |
| workbench cubic K=1282 + 3 Gauss-Newton steps (section 8) | 6406 + 619 | exact; all cases within 1 nm (2 steps: 99.97%, RMSE 0.02 nm) | - |
| workbench cubic OMP, K = 1282 | 6406 | 8.4 nm (median 4.7) | - |
| explicit symbolic expression (PySR) | 4-8 per piston | ~80 nm | 2.6-4.2 nm |
| sparse quadratic polynomial | 155 | 81 nm | 3.1 nm |
| linear | 100 | 100 nm | 6.2 nm |
| MLP ensemble (workbench Part C) | 5 x 204k | 6.7 nm (median 0.3) | - |

- **Acquisition over the full +-400 nm range.** Solve the closed form:
  quadratically convergent Gauss-Newton from a good first estimate, with
  twin and random restarts flagged by the residual. It is exact without
  noise, robust to noise up to a few percent, and calibrated from about 100
  known configurations.
- **Closed loop, small residuals.** Explicit expressions work: a linear map
  below +-50 nm, and compact symbolic expressions (4-8 constants) at
  +-100 nm.
- **What limits both.** The low-order aberrations the piston-only models
  leave out: about 0.6 nm of piston error per nm RMS. Extending the closed
  form to them is the natural next step.